In [ ]:
# notebook for running all the components of the pipeline for reproducing the baseline paper
import os
import matplotlib.pyplot as plt
import numpy as np
import cv2

try:
    from google.colab import drive
    os.chdir('/content')
    drive.mount('/content/drive', force_remount=True)

    !pip install opencv-python
    !pip install joblib
    os.chdir('/content/drive/MyDrive/3dgs_decomposition')
    !pip install -r intr_decomp_3dgs/requirements.txt
    !pip install intr_decomp_3dgs/diff-gaussian-rasterization  --no-build-isolation -v
    !pip install intr_decomp_3dgs/third_party/simple-knn intr_decomp_3dgs/third_party/fused_ssim --no-build-isolation
except ModuleNotFoundError:
    pass  # running locally, already in project root

In [ ]:
scenes = [('hypersim', 'ai_001_001'), ('hypersim', 'ai_002_001'), ('mipnerf360', 'bicycle'), ('mipnerf360', 'garden'), ('mipnerf360', 'room')]
albedo_predictors = ['colourful_iid', 'marigold', 'rgbx']

In [ ]:
from splat.fit_baseline import align_depth_scale, run_baseline_train, render_test_views

In [ ]:
for dataset, scene in scenes:
  align_depth_scale(scene_dir=f'data/{dataset}/{scene}', depths_dir=f'depth_anything_v2/depths/{dataset}/{scene}')

In [ ]:
# run baseline train and render and evaluate test views
for dataset, scene in scenes:
  for predictor in albedo_predictors:
    output_dir = os.path.abspath(f'baseline_train/{predictor}/{dataset}/{scene}')
    albedos_path = os.path.abspath(f'{predictor}/{dataset}/{scene}/albedos')
    depth_paths = os.path.abspath(f'depth_anything_v2/depths/{dataset}/{scene}')
    if dataset == 'mipnerf360':
      source_path = os.path.abspath(f'data/{dataset}/{scene}')
      images_folder = 'images_8'
    elif dataset == 'hypersim':
      source_path = os.path.abspath(f'data/{dataset}/colmap_format/{scene}/cam_00')
      images_folder = 'images'
    else:
      print('Invalid Dataset Name')
      raise KeyboardInterrupt
    os.makedirs(output_dir, exist_ok=True)

    run_baseline_train(
        images_folder=images_folder, # path to /images folder inside source_path
        source_path=source_path, # parent directory to /sparse/0/ and /images folder
        albedos_path=albedos_path, # path to 2D albedo maps for the predictor used
        depth_paths=depth_paths, # path to folder of 16-bit depth pngs
        output_dir=output_dir, # path to where we want the output
        eval=True,
        iterations=10000
    )

    render_test_views(
        source_path=source_path,
        output_dir=output_dir,
        iterations=10000
    )